In [1]:
import csv
import json
import pandas as pd
import itertools
from sentence_transformers import SentenceTransformer, util # For Cosine similarity

/home/dferna3/.conda/envs/env_ft_llama2/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [2]:
# Read results.csv
results_filename = "../data/results.csv"
output_file = "metrics_results"

In [4]:
df = pd.read_csv(results_filename)

# Cosine Similarity

In [15]:
from sentence_transformers import SentenceTransformer, util

In [5]:
ground_truth = f": Placing a stop sign on the roadway shoulder of a highway where vehicles can travel at 65 mph speed, with broken white lane markings is inconsistent with standard traffic regulations. According to the MUTCD Sections 2B.04 (Regulatory Sign Requirements) and 2B.05 (Stop Sign Placement), stop signs are intended for use at intersections, ramps, or specific access points where controlled stops are necessary. Highways are designed for uninterrupted, high-speed traffic flow, and introducing a stop sign in the main travel area disrupts this flow and poses safety risks.The improper placement on the shoulder, rather than at a designated intersection, can lead to confusion among drivers and potential traffic hazards. "

In [6]:
generated_paragraphs = [
    " A stop sign (R1-1) is inappropriate for a highway where the speed limit is 65 mph.  Stop signs are used to regulate traffic at intersections, typically in residential areas or where there's a significant pedestrian or cross-traffic risk.  Placing a stop sign on a highway where the speed limit is 65 mph presents a significant safety hazard and contradicts standard traffic engineering practices.  The presence of a stop sign under these conditions indicates an inconsistency in the scenario.",
]

In [19]:
# Encode ground truth and generated paragraphs
gt_embedding = model.encode(ground_truth, convert_to_tensor=True)
gp_embedding = model.encode(generated_paragraphs, convert_to_tensor=True)

In [20]:
# Compute cosine similarity for each generated paraprah
cos_similarity_scores = util.pytorch_cos_sim(gt_embedding, gp_embedding)

### Cosine Similarity Scores
* Each cosine similarity score is between -1 and 1
* Closer to 1 = higher semantic similarity

In [21]:
#Show results
for idx, score in enumerate(cos_similarity_scores[0]):
    print(f"Generted Paragraph {idx+1}: Similarity Score = {score.item():.4f}")

Generted Paragraph 1: Siimilarity Score = 0.7644


# BERTScore

In [22]:
from bert_score import score

* **Precision**: measures how much of the generated text is sematically similar to the groud truth text
    * Proportion of tokens in the generated text that match (semantically) with tokens in the reference text
    * Scale (0 to 1): higher values mean generated text aligs more closely with the reference in terms of token similarity
* **Recall**: measures how much of the reference text is captured by the generated text
    * Proportion of tokens in the generated text that match (semantically) with tokens in the reference text
    * Scale (0 to 1): higher values mean generated text captures more of the semantic meaning of the refrence text
* **Precision**: harmonic mean of precision and recall. Balances precision and recall to provide a single metric for overall similarity
    * Proportion of tokens in the generated text that match (semantically0 with tokens in the reference text
    * Scale (0 to 1): higher values indicate the generated text both aligns well with the refrence (precision) and captures its meaning (recall)

In [23]:
P, R, F1 = score([ground_truth], generated_paragraphs, lang="en", verbose=True)
print(f"Precision:{P[0]:.4f}")
print(f"Recall:{R[0]:.4f}")
print(f"F1Score:{F1[0]:.4f}")

/home/dferna3/.conda/envs/env_ft_llama2/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


100%|██████████| 1/1 [00:00<00:00, 22.32it/s]


computing greedy matching.


100%|██████████| 1/1 [00:00<00:00, 79.06it/s]

done in 0.06 seconds, 16.45 sentences/sec
Precision:0.8674
Recall:0.8847
F1Score:0.8760


# ROUGE

In [24]:
from rouge_score import rouge_scorer

In [26]:
# Initialize the ROUGE scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

### Metrics
* **ROUGE-1**: Measures unigram (single-word) overlap between the genrated and reference text. Usefo for evaluating the genral similarity of words.
* **ROUGE-2**: Measures bigram (two-word) overlap between texts. Evaluates the fluency and coherence of text generation.
* **ROUGE-L**: Mesures the longest common subsequence (LCS) between texts. Captures sentences structure and grammatical alignment.

Each metric provides
* Precision: proportion of n-grams in the generated text that appears in the reference text
* Recall: proportion of n-grams in the reference text that are captured by the generated text
* F1 Score: Harmonic mean of precision and recall

In [27]:
results = []

In [28]:
# Compute ROUGE scores for each generated text
for gen in generated_paragraphs:
    scores = scorer.score(ground_truth, gen)
    results.append({
        "Generated": gen,
        "ROUGE-1 Precision": scores['rouge1'].precision,
        "ROUGE-1 Recall": scores['rouge1'].recall,
        "ROUGE-1 F1": scores['rouge1'].fmeasure,
        "ROUGE-2 Precision": scores['rouge2'].precision,
        "ROUGE-2 Recall": scores['rouge2'].recall,
        "ROUGE-2 F1": scores['rouge2'].fmeasure,
        "ROUGE-L Precision": scores['rougeL'].precision,
        "ROUGE-L Recall": scores['rougeL'].recall,
        "ROUGE-L F1": scores['rougeL'].fmeasure,
    })

In [31]:
# Display the results
import pandas as pd
results_df = pd.DataFrame(results)

Ranges: 
* 0: No overlap between the generated text and the reference text.
* 0-0.3: Little to no overlap between the generated text and the reference.Generated texts may be off-topic, irrelevant, or highly dissimilar to the reference.
* 0.3-0.6: Moderate overlap between the generated and reference texts. Generated texts capture some key information but may miss details or introduce irrelevant content.
* 0.6-1.0: High overlap, indicating the generated text closely matches the reference text. Higher precision implies accuracy, while higher recall implies comprehensiveness.
* 1: Perfect overlap between the generated text and the reference text.

In [32]:
# Compute average ROUGE scores
avg_rouge1_f1 = results_df["ROUGE-1 F1"].mean()
avg_rouge2_f1 = results_df["ROUGE-2 F1"].mean()
avg_rougeL_f1 = results_df["ROUGE-L F1"].mean()

print(f"Average ROUGE-1 F1: {avg_rouge1_f1:.4f}")
print(f"Average ROUGE-2 F1: {avg_rouge2_f1:.4f}")
print(f"Average ROUGE-L F1: {avg_rougeL_f1:.4f}")

Average ROUGE-1 F1: 0.4870
Average ROUGE-2 F1: 0.1675
Average ROUGE-L F1: 0.2591


# BLEU

In [1]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

### BLEU Score Scale
* 0.0: No overlap between the generated and reference text.
* 0.0 - 0.2: Very low overlap (generated text is unrelated or highly paraphrased).
* 0.2 - 0.4: Moderate overlap (some words or phrases match, but large differences remain).
* 0.4 - 0.7: Good overlap (generated text is close to the reference, but may have slight phrasing differences).
* 0.7 - 1.0: High overlap (generated text closely mirrors the reference).
* 1.0: Perfect n-gram match between the generated and reference text.

In [7]:
results = []
for gen in generated_paragraphs:
    candidate = gen.split()  # Tokenize the generated text
    bleu_score = sentence_bleu(ground_truth, candidate, smoothing_function=SmoothingFunction().method1)
    results.append({
        "Generated Text": gen,
        "BLEU Score": f"{bleu_score:.4f}"
    })
for idx, result in enumerate(results):
    print(f"Generated Paragraph {idx+1}: {result['BLEU Score']}")

Generated Paragraph 1: 0.0028
